# 08.01 — Common Data Splits

Build and inspect the chronological expanding-window validation folds and the untouched 2025 holdout.

In [1]:
# Import libraries
from pathlib import Path
import sys

In [2]:
# Define the root directory of the project
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = PROJECT_ROOT / "configs" / "modeling_foundation.yaml"
CONFIG_PATH

WindowsPath('e:/jcuenca/OneDrive - GUSCanada/5toTerm/01_Capstone/Codigo/ontario-electricity-peak-risk/configs/modeling_foundation.yaml')

In [3]:
# Import module to manage modeling configuration, feature datasets, and directories
from src.ontario_peak_risk.modeling.common import (
    load_modeling_config,
    load_feature_dataset,
    ensure_modeling_directories,
)

In [4]:
# Import odule to manage modeling load forecasting targets
from src.ontario_peak_risk.modeling.common import (
    load_forecast_targets,
)

In [5]:
# Load the modeling configuration, feature dataset, and ensure necessary directories exist
CONFIG, _ = load_modeling_config(CONFIG_PATH)
REPORTS_DIR, DOCS_DIR, OUTPUTS_DIR = ensure_modeling_directories(CONFIG, PROJECT_ROOT)
feature_dataset = load_feature_dataset(CONFIG, PROJECT_ROOT)
feature_dataset.shape

(262944, 100)

The 2025 holdout must remain untouched during model development and model selection.

In [6]:
# Import module to manage data splits for modeling
from src.ontario_peak_risk.modeling.splits import (
    build_validation_folds,
    build_final_holdout_fold,
    split_summary,
)

In [7]:
# Build validation folds and final holdout fold based on the configuration
forecast_targets = load_forecast_targets(CONFIG, PROJECT_ROOT)
validation_folds = build_validation_folds(CONFIG)
final_holdout = build_final_holdout_fold(CONFIG)
all_folds = [*validation_folds, final_holdout]

In [8]:
# Display horizons
horizons = CONFIG["modeling"]["forecasting"]["horizons"]
horizons

24

In [ ]:
# Display split_report
split_report = split_summary(
    forecast_targets,
    all_folds,
    time_column="forecast_origin",
    group_column=CONFIG["modeling"]["time"]["group_column"],
    max_horizon_hours=horizons,
)

split_report

,fold,role,configured_train_start,configured_train_end,safe_train_origin_end,train_rows,train_first_origin,train_last_origin,train_fsas,configured_evaluation_start,configured_evaluation_end,safe_evaluation_origin_end,evaluation_rows,evaluation_first_origin,evaluation_last_origin,evaluation_fsas,max_horizon_hours
0,fold_2023,validation,2021-01-01,2022-12-31 23:00:00,2022-12-30 23:00:00,104976,2021-01-01,2022-12-30 23:00:00,6,2023-01-01,2023-12-31 23:00:00,2023-12-30 23:00:00,52416,2023-01-01,2023-12-30 23:00:00,6,24
1,fold_2024,validation,2021-01-01,2023-12-31 23:00:00,2023-12-30 23:00:00,157536,2021-01-01,2023-12-30 23:00:00,6,2024-01-01,2024-12-31 23:00:00,2024-12-30 23:00:00,52560,2024-01-01,2024-12-30 23:00:00,6,24
2,test_2025,test,2021-01-01,2024-12-31 23:00:00,2024-12-30 23:00:00,210240,2021-01-01,2024-12-30 23:00:00,6,2025-01-01,2025-12-31 23:00:00,2025-12-30 23:00:00,52416,2025-01-01,2025-12-30 23:00:00,6,24


In [11]:
# Generate a summary report of the data splits with time_column config
split_report = split_summary(
    feature_dataset,
    all_folds,
    time_column=CONFIG["modeling"]["time"]["time_column"],
    group_column=CONFIG["modeling"]["time"]["group_column"],
)

split_report

,fold,role,configured_train_start,configured_train_end,safe_train_origin_end,train_rows,train_first_origin,train_last_origin,train_fsas,configured_evaluation_start,configured_evaluation_end,safe_evaluation_origin_end,evaluation_rows,evaluation_first_origin,evaluation_last_origin,evaluation_fsas,max_horizon_hours
0,fold_2023,validation,2021-01-01,2022-12-31 23:00:00,2022-12-31 23:00:00,105120,2021-01-01,2022-12-31 23:00:00,6,2023-01-01,2023-12-31 23:00:00,2023-12-31 23:00:00,52560,2023-01-01,2023-12-31 23:00:00,6,0
1,fold_2024,validation,2021-01-01,2023-12-31 23:00:00,2023-12-31 23:00:00,157680,2021-01-01,2023-12-31 23:00:00,6,2024-01-01,2024-12-31 23:00:00,2024-12-31 23:00:00,52704,2024-01-01,2024-12-31 23:00:00,6,0
2,test_2025,test,2021-01-01,2024-12-31 23:00:00,2024-12-31 23:00:00,210384,2021-01-01,2024-12-31 23:00:00,6,2025-01-01,2025-12-31 23:00:00,2025-12-31 23:00:00,52560,2025-01-01,2025-12-31 23:00:00,6,0
